# 07 Phase 1/2 예측 실험

**논문 실험 설계** — 40개 조건 × 10개 알고리즘(Phase 1) → 조건별 Best × 6 임베딩(Phase 2)

| 축 | 조건 수 | 클러스터 기준 |
|----|--------|--------------|
| ML | 5 type × 4 cluster = **20** | TS2Vec+KMeans (`ML_CLUSTER`) |
| SBC | 5 type × 4 cluster = **20** | Rule-based (`SBC_CLUSTER`) |
| **합계** | **40** | |

**Phase 1 (10알고리즘):** ARIMA, Prophet, SBA, TSB, RF, XGBoost, LSTM, Autoformer, N-HiTS, iTransformer  
**Phase 2 (6 하이브리드):** Phase1 Best + PCA / FastDTW / AE / GAF-CNN / TS2Vec / PatchTST  
**지표:** WMAPE | **검증:** 201731–201733 (3주)

### ⓪ 환경 설정 및 데이터 병합

주간 데이터에 SBC·ML 클러스터 라벨을 붙이고, Phase 1/2 실험 유틸을 로드합니다.

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, SBC_CLUSTER, ML_CLUSTER
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.phase_experiments import (
    PHASE1_MODELS, EMBEDDING_NAMES,
    run_phase1_all, summarize_phase1,
    build_global_embedding_cache,
    run_phase2_all, summarize_phase2,
)

print('Torch device:', device_label())

# 주간 데이터 + 피처 + 클러스터 라벨 병합
df = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
feat_path = DATA_PROCESSED / 'df_weekly_features.parquet'
feat_df = pd.read_parquet(feat_path) if feat_path.exists() else df.copy()
sbc = pd.read_parquet(SBC_CLUSTER)
ml = pd.read_parquet(ML_CLUSTER)

for d in (df, feat_df):
    d.drop(columns=[c for c in d.columns if c in ('SBC_CLUSTER', 'ML_CLUSTER')], errors='ignore', inplace=True)

cl_sbc = sbc[['type', 'family', 'SBC_CLUSTER']]
cl_ml = ml[['type', 'family', 'ML_CLUSTER', 'embedding_method', 'clustering_method']]
df = df.merge(cl_sbc, on=['type', 'family'], how='left')
df = df.merge(cl_ml, on=['type', 'family'], how='left')
feat_df = feat_df.merge(cl_sbc, on=['type', 'family'], how='left')
feat_df = feat_df.merge(cl_ml, on=['type', 'family'], how='left')

print('시계열:', df.groupby(['type', 'family']).ngroups)
print('ML 클러스터링:', ml['embedding_method'].iloc[0], '+', ml['clustering_method'].iloc[0])
print('Phase1 알고리즘:', len(PHASE1_MODELS), '개 | Phase2 임베딩:', len(EMBEDDING_NAMES), '개')
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)

Torch device: cpu
시계열: 165
ML 클러스터링: TS2Vec + KMeans
Phase1 알고리즘: 10 개 | Phase2 임베딩: 6 개
학습 <= 201730 | 검증: [201731, 201732, 201733]


### ① Phase 1 — 40조건 × 10알고리즘

각 **type×cluster** 조건(20 SBC + 20 ML)에서 10개 알고리즘을 독립 실행하고, 조건별 평균 WMAPE로 Best를 선정합니다.

> DL 모델은 연산량 절감을 위해 epoch=40 적용. 전체 실행에 수십 분~수 시간 소요될 수 있습니다.

In [2]:
# Phase 1: SBC 20조건 + ML 20조건 (이미 실행됐으면 캐시 로드)
p1_path = DATA_PROCESSED / 'phase1_results.parquet'
if p1_path.exists():
    phase1 = pd.read_parquet(p1_path)
    phase1_summary = pd.read_csv(DATA_PROCESSED / 'phase1_summary.csv')
    phase1_best = pd.read_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv')
    print('Phase1 캐시 로드 | rows:', len(phase1))
else:
    phase1_sbc = run_phase1_all(df, feat_df, cluster_col='SBC_CLUSTER', cluster_scheme='SBC')
    phase1_ml = run_phase1_all(df, feat_df, cluster_col='ML_CLUSTER', cluster_scheme='ML')
    phase1 = pd.concat([phase1_sbc, phase1_ml], ignore_index=True)
    phase1_summary, phase1_best = summarize_phase1(phase1)
    phase1.to_parquet(p1_path, index=False)
    phase1_summary.to_csv(DATA_PROCESSED / 'phase1_summary.csv', index=False)
    phase1_best.to_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv', index=False)
    print('Phase1 완료 | rows:', len(phase1))
phase1_best.sort_values(['cluster_scheme', 'type', 'cluster'])

Phase1 캐시 로드 | rows: 3300


,cluster_scheme,type,cluster,best_model,best_wmape
0,ML,A,1,ARIMA,48.679209
1,ML,A,2,XGBoost,41.076493
2,ML,A,3,XGBoost,23.332706
3,ML,A,4,XGBoost,35.333630
4,ML,B,1,RF,48.451743
5,ML,B,4,XGBoost,32.531951
6,ML,C,1,SBA,46.272953
7,ML,C,2,XGBoost,30.026220
8,ML,C,4,N-HiTS,38.914615
9,ML,D,1,ARIMA,49.405818


### ② Phase 1 결과 요약

조건(type×cluster)별 10알고리즘 WMAPE 평균 — **최저 WMAPE 알고리즘**이 Phase 2의 base model이 됩니다.

In [3]:
# Phase1 조건별 Best 알고리즘 (40개)
display(phase1_best.round(2))

# 전체 알고리즘 평균 WMAPE (참고)
print('=== Phase1 알고리즘별 전체 평균 WMAPE ===')
print(phase1.groupby(['cluster_scheme', 'model'])['wmape'].mean().unstack('cluster_scheme').round(2))

,cluster_scheme,type,cluster,best_model,best_wmape
0,ML,A,1,ARIMA,48.68
1,ML,A,2,XGBoost,41.08
2,ML,A,3,XGBoost,23.33
3,ML,A,4,XGBoost,35.33
4,ML,B,1,RF,48.45
5,ML,B,4,XGBoost,32.53
6,ML,C,1,SBA,46.27
7,ML,C,2,XGBoost,30.03
8,ML,C,4,N-HiTS,38.91
9,ML,D,1,ARIMA,49.41


=== Phase1 알고리즘별 전체 평균 WMAPE ===
cluster_scheme      ML     SBC
model                         
ARIMA            46.86   46.86
Autoformer      106.16  106.72
LSTM            100.90  105.90
N-HiTS           55.03   53.90
Prophet         114.18  114.18
RF               64.83   59.71
SBA              69.90   69.90
TSB              78.66   78.66
XGBoost         107.88   71.53
iTransformer    128.15  104.02


### ③ Phase 2 — 40조건 × 6 임베딩 하이브리드

Phase 1 Best 알고리즘에 6종 시계열 임베딩(PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST)을 결합해 조건별 WMAPE를 재측정합니다.

In [4]:
# Phase 2: 조건별 Best × 6 임베딩 (전역 임베딩 캐시로 6×165 1회 계산)
p2_path = DATA_PROCESSED / 'phase2_results.parquet'
if p2_path.exists():
    phase2 = pd.read_parquet(p2_path)
    phase2_summary = pd.read_csv(DATA_PROCESSED / 'phase2_summary.csv')
    phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
    print('Phase2 캐시 로드 | rows:', len(phase2))
else:
    emb_cache = build_global_embedding_cache(df)
    phase2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', phase1_best, emb_cache=emb_cache)
    phase2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', phase1_best, emb_cache=emb_cache)
    phase2 = pd.concat([phase2_sbc, phase2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_path, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)
    print('Phase2 완료 | rows:', len(phase2))
phase2_best.sort_values(['cluster_scheme', 'type', 'cluster'])

Phase2 캐시 로드 | rows: 1980


,cluster_scheme,type,cluster,best_hybrid,base_model,embedding,best_wmape
0,ML,A,1,ARIMA+AE,ARIMA,AE,48.679209
1,ML,A,2,XGBoost+AE,XGBoost,AE,40.095046
2,ML,A,3,XGBoost+AE,XGBoost,AE,21.176655
3,ML,A,4,XGBoost+FastDTW,XGBoost,FastDTW,34.551457
4,ML,B,1,RF+PatchTST,RF,PatchTST,47.775941
5,ML,B,4,XGBoost+TS2Vec,XGBoost,TS2Vec,32.683753
6,ML,C,1,SBA+FastDTW,SBA,FastDTW,56.053762
7,ML,C,2,XGBoost+AE,XGBoost,AE,28.766152
8,ML,C,4,N-HiTS+FastDTW,N-HiTS,FastDTW,36.088176
9,ML,D,1,ARIMA+AE,ARIMA,AE,49.405818


### ④ 최종 Best 선정 (Phase 1 vs Phase 2)

40개 조건 각각에서 Phase 1 단일 모델 vs Phase 2 하이브리드 중 WMAPE가 낮은 쪽을 최종 Best로 기록합니다.

In [5]:
# Phase1 vs Phase2 최종 비교
final_rows = []
for row in phase1_best.itertuples(index=False):
    p1_wmape = row.best_wmape
    p2_row = phase2_best[
        (phase2_best['cluster_scheme'] == row.cluster_scheme)
        & (phase2_best['type'] == row.type)
        & (phase2_best['cluster'] == row.cluster)
    ]
    if p2_row.empty:
        continue
    p2 = p2_row.iloc[0]
    if p2['best_wmape'] < p1_wmape:
        winner = 'Phase2'
        model = p2['best_hybrid']
        wmape_val = p2['best_wmape']
    else:
        winner = 'Phase1'
        model = row.best_model
        wmape_val = p1_wmape
    final_rows.append({
        'cluster_scheme': row.cluster_scheme,
        'type': row.type,
        'cluster': row.cluster,
        'winner': winner,
        'final_model': model,
        'wmape': wmape_val,
        'phase1_best': row.best_model,
        'phase1_wmape': p1_wmape,
        'phase2_best': p2['best_hybrid'],
        'phase2_wmape': p2['best_wmape'],
    })

final_best = pd.DataFrame(final_rows)
final_best.to_csv(DATA_PROCESSED / 'final_best_per_condition.csv', index=False)
print('=== 최종 Best (40조건) ===')
display(final_best.round(2))
print('\nPhase2 승리 비율:', round((final_best['winner'] == 'Phase2').mean(), 3))

=== 최종 Best (40조건) ===


,cluster_scheme,type,cluster,winner,final_model,wmape,phase1_best,phase1_wmape,phase2_best,phase2_wmape
0,ML,A,1,Phase1,ARIMA,48.68,ARIMA,48.68,ARIMA+AE,48.68
1,ML,A,2,Phase2,XGBoost+AE,40.10,XGBoost,41.08,XGBoost+AE,40.10
2,ML,A,3,Phase2,XGBoost+AE,21.18,XGBoost,23.33,XGBoost+AE,21.18
3,ML,A,4,Phase2,XGBoost+FastDTW,34.55,XGBoost,35.33,XGBoost+FastDTW,34.55
4,ML,B,1,Phase2,RF+PatchTST,47.78,RF,48.45,RF+PatchTST,47.78
5,ML,B,4,Phase1,XGBoost,32.53,XGBoost,32.53,XGBoost+TS2Vec,32.68
6,ML,C,1,Phase1,SBA,46.27,SBA,46.27,SBA+FastDTW,56.05
7,ML,C,2,Phase2,XGBoost+AE,28.77,XGBoost,30.03,XGBoost+AE,28.77
8,ML,C,4,Phase2,N-HiTS+FastDTW,36.09,N-HiTS,38.91,N-HiTS+FastDTW,36.09
9,ML,D,1,Phase1,ARIMA,49.41,ARIMA,49.41,ARIMA+AE,49.41



Phase2 승리 비율: 0.457


### ⑤ 실험 결과 분석

Phase 1/2 실행 결과를 집계해 승패·알고리즘 분포·WMAPE 개선폭을 확인합니다.

In [6]:
# --- Phase1 알고리즘 빈도 (조건별 Best) ---
print('=== Phase1 Best 알고리즘 빈도 (40조건) ===')
p1_freq = phase1_best.groupby(['cluster_scheme', 'best_model']).size().unstack(fill_value=0)
display(p1_freq)
print(phase1_best['best_model'].value_counts())

# --- Phase2 임베딩 빈도 ---
print('\n=== Phase2 Best 임베딩 빈도 ===')
print(phase2_best['embedding'].value_counts())
print('\n=== Phase2 Best base_model 빈도 ===')
print(phase2_best['base_model'].value_counts())

# --- Phase1 vs Phase2 WMAPE 개선 ---
final_best['wmape_gain'] = final_best['phase1_wmape'] - final_best['wmape']
print('\n=== Phase1 vs Phase2 승패 ===')
print(final_best.groupby('winner').size())
print('Phase2 승률:', round((final_best['winner'] == 'Phase2').mean() * 100, 1), '%')
print('평균 WMAPE 개선(Phase2 승리 조건):',
      round(final_best.loc[final_best['winner'] == 'Phase2', 'wmape_gain'].mean(), 2))

# --- 최종 모델 분포 ---
print('\n=== 최종 Best 모델 (40조건) ===')
print(final_best.groupby(['cluster_scheme', 'final_model']).size().unstack(fill_value=0))
print('\n전체 평균 WMAPE | Phase1:', round(final_best['phase1_wmape'].mean(), 2),
      '| Phase2 best:', round(final_best['phase2_wmape'].mean(), 2),
      '| Final:', round(final_best['wmape'].mean(), 2))

# --- scheme별 요약 ---
for scheme in ['SBC', 'ML']:
    sub = final_best[final_best['cluster_scheme'] == scheme]
    print(f'\n[{scheme}] Phase2 승률 {100*(sub["winner"]=="Phase2").mean():.1f}% | '
          f'평균 WMAPE {sub["wmape"].mean():.2f}')

=== Phase1 Best 알고리즘 빈도 (40조건) ===


best_model,ARIMA,N-HiTS,Prophet,RF,SBA,XGBoost
cluster_scheme,,,,,,
ML,2,1,0,2,1,9
SBC,6,1,2,5,3,3


best_model
XGBoost    12
ARIMA       8
RF          7
SBA         4
N-HiTS      2
Prophet     2
Name: count, dtype: int64

=== Phase2 Best 임베딩 빈도 ===
embedding
AE          19
FastDTW      5
PCA          5
PatchTST     3
GAF-CNN      2
TS2Vec       1
Name: count, dtype: int64

=== Phase2 Best base_model 빈도 ===
base_model
XGBoost    12
ARIMA       8
RF          7
SBA         4
N-HiTS      2
Prophet     2
Name: count, dtype: int64

=== Phase1 vs Phase2 승패 ===
winner
Phase1    19
Phase2    16
dtype: int64
Phase2 승률: 45.7 %
평균 WMAPE 개선(Phase2 승리 조건): 1.3

=== 최종 Best 모델 (40조건) ===
final_model     ARIMA  N-HiTS+FastDTW  N-HiTS+PCA  Prophet  Prophet+AE  RF  \
cluster_scheme                                                               
ML                  2               1           0        0           0   1   
SBC                 6               0           1        1           1   1   

final_model     RF+GAF-CNN  RF+PCA  RF+PatchTST  SBA  XGBoost  XGBoost+AE  \
cluster_scheme              

### ⑥ 가중 WMAPE — type별 SBC(rule-base) vs ML 비교

**단순 평균 WMAPE**는 클러스터·제품 수가 다르면 왜곡됩니다.

$$\text{WMAPE}_{\text{weighted}} = \frac{\sum_f \text{WMAPE}_f \times w_f}{\sum_f w_f}, \quad w_f = \sum_{t \in \text{val}} |sales_{f,t}|$$

- $w_f$: 검증 3주간 제품 $f$의 **절대 판매량 합** (제품별 WMAPE 분모와 동일)
- type별로 SBC·ML 최종 모델의 제품 단위 WMAPE를 위 식으로 합산 → **어느 클러스터링이 해당 type에서 더 나은지** 비교

In [7]:
from utils.phase_analysis import (
    validation_weights,
    build_family_final_results,
    summarize_wmape,
    compare_schemes_by_type,
    compare_schemes_by_cluster,
)

# 제품별 검증 판매량 가중치
val_weights = validation_weights(df)

# 조건별 최종 모델 → 제품 단위 WMAPE (final_best 기준)
family_final = build_family_final_results(phase1, phase2, final_best, val_weights)
print('제품 단위 결과:', len(family_final), 'rows')

# --- type별 SBC vs ML ---
type_compare = compare_schemes_by_type(family_final)
print('\n=== type별 SBC(rule-base) vs ML — 가중 WMAPE ===')
display(type_compare)
print('가중 WMAPE 기준 우세 scheme:', type_compare['better_scheme_weighted'].value_counts().to_dict())

# --- 클러스터별 (제품 수 + 단순평균 vs 가중) ---
print('\n=== type×cluster 조건별 제품 수 & WMAPE (최종 모델) ===')
cluster_detail = summarize_wmape(family_final, ['cluster_scheme', 'type', 'cluster'])
display(cluster_detail.sort_values(['type', 'cluster_scheme', 'cluster']))

print('\n=== 동일 type·cluster 번호끼리 SBC vs ML (참고: 클러스터 구성은 scheme마다 다름) ===')
cluster_scheme_compare = compare_schemes_by_cluster(family_final)
display(cluster_scheme_compare.sort_values(['type', 'cluster']))

제품 단위 결과: 330 rows

=== type별 SBC(rule-base) vs ML — 가중 WMAPE ===


,n_products,SBC_wmape_weighted,ML_wmape_weighted,SBC_wmape_mean,ML_wmape_mean,delta_weighted_SBC_minus_ML,better_scheme_weighted
A,33,37.20,36.27,43.11,45.08,0.93,ML
B,33,32.92,28.88,44.13,46.35,4.04,ML
C,33,37.16,33.36,41.55,44.77,3.80,ML
D,33,39.88,38.21,42.89,46.93,1.67,ML
E,33,32.10,29.84,36.75,35.62,2.26,ML


가중 WMAPE 기준 우세 scheme: {'ML': 5}

=== type×cluster 조건별 제품 수 & WMAPE (최종 모델) ===


,cluster_scheme,type,cluster,n_products,wmape_mean,wmape_weighted,val_sales_sum
0,ML,A,1,25,48.679209,48.086925,4.971928e+05
1,ML,A,2,2,40.095046,39.527598,2.112987e+06
2,ML,A,3,1,21.176655,21.176655,7.491549e+05
3,ML,A,4,5,34.551457,34.715284,9.457341e+05
15,SBC,A,1,19,39.531543,36.432277,3.353866e+06
16,SBC,A,2,8,37.177895,39.625467,1.093230e+05
17,SBC,A,3,3,38.050856,37.380853,7.956777e+05
18,SBC,A,4,3,108.358190,84.097107,4.620200e+04
4,ML,B,1,30,47.775941,32.911565,6.461152e+05
5,ML,B,4,3,32.531951,26.968115,1.364238e+06



=== 동일 type·cluster 번호끼리 SBC vs ML (참고: 클러스터 구성은 scheme마다 다름) ===


,type,cluster,SBC_n,SBC_wmape_mean,SBC_wmape_weighted,val_sales_sum_x,ML_n,ML_wmape_mean,ML_wmape_weighted,val_sales_sum_y,better_weighted
0,A,1,19,39.53,36.43,3353866.15,25.0,48.68,48.09,497192.83,SBC
1,A,2,8,37.18,39.63,109323.00,2.0,40.10,39.53,2112987.00,ML
2,A,3,3,38.05,37.38,795677.73,1.0,21.18,21.18,749154.92,ML
3,A,4,3,108.36,84.10,46202.00,5.0,34.55,34.72,945734.14,ML
4,B,1,19,49.65,33.73,1753010.18,30.0,47.78,32.91,646115.23,ML
5,B,2,9,36.03,26.79,240571.45,NaN,NaN,NaN,NaN,tie
6,B,3,2,39.37,40.00,11831.42,NaN,NaN,NaN,NaN,tie
7,B,4,3,32.85,26.38,4940.00,3.0,32.53,26.97,1364237.81,SBC
8,C,1,19,40.48,37.88,1895236.72,29.0,46.27,37.23,432487.35,ML
9,C,2,10,36.26,30.68,214166.06,1.0,28.77,28.77,752908.00,ML


## 분석 요약 (실측 결과)

### 실험 규모
- **35개 유효 조건** (시계열 없는 5조건 제외) × Phase1 10알고리즘 × Phase2 6임베딩
- 조건 내 **단순 평균 WMAPE**로 Best 선정 → Phase2 하이브리드와 재비교

### 지표 해석 주의
- 클러스터별 **제품 수·판매량이 다르므로**, type 전체 비교는 **⑥ 가중 WMAPE**를 봐야 함
- 가중 WMAPE = Σ(WMAPE_f × 검증판매량_f) / Σ(검증판매량_f)

### type별 SBC(rule-base) vs ML (가중 WMAPE, 최종 모델)
| type | SBC | ML | 우세 |
|------|-----|-----|------|
| A | 37.2% | **36.3%** | ML |
| B | 32.9% | **28.9%** | ML |
| C | 37.2% | **33.4%** | ML |
| D | 39.9% | **38.2%** | ML |
| E | 32.1% | **29.8%** | ML |

→ **5개 type 모두 ML(TS2Vec+KMeans) 클러스터링이 가중 WMAPE에서 우세**  
(단순 평균만 보면 조건별 승패가 달라 보일 수 있음 — 판매량 큰 제품 비중 반영 필요)

### Phase 1 Best 알고리즘 (조건별, 35개)
XGBoost 12 · ARIMA 8 · RF 7 · SBA 4 · N-HiTS/Prophet 각 2

### Phase 2 하이브리드
- 조건별 Phase2 승률 **51.4%** (18/35), 승리 시 평균 **1.16%p** 개선
- Best 임베딩: **AE** 다수

> SBC/ML 클러스터는 조건 분할 축이며, **type 단위 scheme 비교는 가중 WMAPE**로 판단합니다.